# Phase 3 — Ground Truth Manchester

**Objetivo:** Asignar niveles de prioridad Manchester (C1-C5) a los 272 casos del dataset OSCE.

**Estrategia:**
1. Prior por especialidad (regla simple, punto de partida)
2. LLM (Claude Haiku) sugiere nivel por caso → `nivel_llm_sugerido`
3. Prior se usa como fallback si LLM falla
4. `nivel_manchester` = nivel definitivo para entrenamiento
5. `revision_humana=False` por defecto — se puede corregir manualmente

**El LLM NO es la decisión final.** Es un asistente de etiquetado. El modelo ML es el núcleo del sistema.

---

In [ ]:
import sys
from pathlib import Path

# Ensure root is in path
ROOT = Path().resolve().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import warnings
warnings.filterwarnings('ignore')

print(f'Root: {ROOT}')

## 1. Cargar dataset Phase 1

In [ ]:
INPUT_CSV = ROOT / 'data' / 'processed' / 'dataset_exploracion_fase1.csv'
df_fase1 = pd.read_csv(INPUT_CSV)
print(f'Shape: {df_fase1.shape}')
print(f'Columnas: {list(df_fase1.columns)}')
df_fase1.head(3)

## 2. Prior por especialidad

Antes de invocar el LLM, establecemos un prior basado en la especialidad clínica.
Esto sirve como fallback y como punto de comparación con la sugerencia del LLM.

In [ ]:
SPECIALTY_PRIOR = {
    'RES': 'C3',  # respiratorio — rango amplio; LLM distingue
    'MSK': 'C4',  # musculoesquelético — generalmente menor
    'GAS': 'C3',  # gastrointestinal
    'CAR': 'C2',  # cardiológico — conservador por seguridad
    'DER': 'C4',  # dermatológico
    'GEN': 'C3',  # general
}

df_fase1['prior_especialidad'] = df_fase1['grupo_clinico'].map(SPECIALTY_PRIOR)

print('Distribución del prior por especialidad:')
print(df_fase1.groupby(['grupo_clinico', 'prior_especialidad']).size().to_string())

## 3. Ejecutar pipeline completo de Ground Truth

Llamamos al script principal. Usa checkpoints cada 10 casos (reanudable).

**Opciones:**
- `--limit 5` para probar con 5 casos
- `--skip-llm` para saltar llamadas LLM (solo prior)
- `--skip-minio` si el stack Docker no está levantado
- `--skip-postgres` si el stack Docker no está levantado

In [ ]:
# Test con 5 casos primero
from src.ground_truth_fase3 import procesar_dataset

df_test = procesar_dataset(
    limit=5,
    skip_minio=True,
    skip_postgres=True,
)
df_test[['id_caso', 'grupo_clinico', 'nivel_llm_sugerido', 'nivel_manchester', 'confianza_llm', 'metodo_label']]

In [ ]:
# PRODUCCIÓN: procesar los 272 casos
# Descomenta cuando estés listo. Tarda ~5 min con Haiku.

# df_maestro = procesar_dataset(
#     skip_minio=False,   # cambiar a True si Docker no está activo
#     skip_postgres=False,
# )

## 4. Análisis del Dataset Maestro

Ejecutar esta sección después de generar `data/master/dataset_maestro.csv`.

In [ ]:
MAESTRO_CSV = ROOT / 'data' / 'master' / 'dataset_maestro.csv'

if MAESTRO_CSV.exists():
    df = pd.read_csv(MAESTRO_CSV)
    print(f'Dataset maestro: {df.shape}')
    print(f'Columnas: {list(df.columns)}')
    df.head(3)
else:
    print(f'Archivo no encontrado: {MAESTRO_CSV}')
    print('Ejecuta primero la celda de producción.')

In [ ]:
if MAESTRO_CSV.exists():
    # Distribución Manchester
    COLORES_MTS = {'C1': '#e74c3c', 'C2': '#e67e22', 'C3': '#f1c40f',
                   'C4': '#2ecc71', 'C5': '#9b59b6'}
    
    dist = df['nivel_manchester'].value_counts().sort_index()
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Bar chart distribución Manchester
    colors = [COLORES_MTS.get(n, '#95a5a6') for n in dist.index]
    axes[0].bar(dist.index, dist.values, color=colors, edgecolor='white', linewidth=0.5)
    for i, (nivel, count) in enumerate(dist.items()):
        axes[0].text(i, count + 1, f'{count}\n({count/len(df)*100:.1f}%)',
                     ha='center', va='bottom', fontsize=9)
    axes[0].set_title('Distribución Manchester (dataset maestro)', fontweight='bold')
    axes[0].set_xlabel('Nivel Manchester')
    axes[0].set_ylabel('Número de casos')
    axes[0].set_ylim(0, dist.max() * 1.15)
    
    # Heatmap especialidad × nivel
    cross = pd.crosstab(df['grupo_clinico'], df['nivel_manchester'])
    for nivel in ['C1','C2','C3','C4','C5']:
        if nivel not in cross.columns:
            cross[nivel] = 0
    cross = cross[['C1','C2','C3','C4','C5']]
    
    im = axes[1].imshow(cross.values, cmap='YlOrRd', aspect='auto')
    axes[1].set_xticks(range(5))
    axes[1].set_xticklabels(['C1','C2','C3','C4','C5'])
    axes[1].set_yticks(range(len(cross)))
    axes[1].set_yticklabels(cross.index)
    for i in range(len(cross)):
        for j in range(5):
            axes[1].text(j, i, str(cross.values[i,j]), ha='center', va='center', fontsize=10)
    axes[1].set_title('Especialidad × Nivel Manchester', fontweight='bold')
    axes[1].set_xlabel('Nivel Manchester')
    axes[1].set_ylabel('Especialidad clínica')
    plt.colorbar(im, ax=axes[1], shrink=0.8)
    
    plt.tight_layout()
    
    FIGURES_DIR = ROOT / 'reports' / 'figures'
    FIGURES_DIR.mkdir(parents=True, exist_ok=True)
    fig_path = FIGURES_DIR / 'distribucion_manchester_fase3.png'
    plt.savefig(fig_path, dpi=150, bbox_inches='tight')
    print(f'Figura guardada: {fig_path}')
    plt.show()

In [ ]:
if MAESTRO_CSV.exists():
    print('=== Calidad del Ground Truth ===')
    print(f"Total casos: {len(df)}")
    print(f"Con nivel LLM sugerido: {df['nivel_llm_sugerido'].notna().sum()}")
    print(f"Fallback a prior especialidad: {df['nivel_llm_sugerido'].isna().sum()}")
    print(f"Confianza LLM media: {df['confianza_llm'].mean():.3f}")
    print(f"Confianza LLM < 0.5: {(df['confianza_llm'] < 0.5).sum()} casos")
    print(f"Con revisión humana: {df['revision_humana'].sum()} (todos False por defecto)")
    print()
    print('Distribución por método de etiquetado:')
    print(df['metodo_label'].value_counts().to_string())

## 5. Revisión manual (opcional)

Para corregir etiquetas manualmente: editar `data/master/dataset_maestro.csv` directamente.
Cambiar `revision_humana=True` y `nivel_manchester` al nivel correcto.
Mantener `nivel_llm_sugerido` original para trazabilidad.

In [ ]:
# Casos con baja confianza LLM (candidatos para revisión humana)
if MAESTRO_CSV.exists():
    baja_confianza = df[df['confianza_llm'] < 0.6].sort_values('confianza_llm')
    print(f'Casos con confianza < 0.6: {len(baja_confianza)}')
    if len(baja_confianza) > 0:
        display_cols = ['id_caso', 'grupo_clinico', 'nivel_llm_sugerido',
                        'nivel_manchester', 'confianza_llm', 'razon_llm']
        baja_confianza[display_cols].head(20)

---
## Resumen Phase 3

| Artefacto | Ubicación |
|-----------|----------|
| Dataset maestro | `data/master/dataset_maestro.csv` |
| Figura distribución | `reports/figures/distribucion_manchester_fase3.png` |
| MinIO | `minio://triageia-processed/ground_truth/dataset_maestro.csv` |
| Decisión documentada | `docs/decisions/03_ground_truth.md` |

**Próximo paso:** Phase 4 — Limpieza y NER (`/triageia-phase fase-4`)